# IUCN Ecosystem Red List Assessment - Data Processing

Processes raw RLE assessment data to produce:
1. A GeoJSON layer for Mapbox upload (replacing `globalmangrovewatch.20gft2fx`, source-layer `ecoregions_data`)
2. `ecoregions.csv` for backoffice widget table
3. `ecoregion_reports.csv` for backoffice reports table

### Input files (in `../../data/raw/`):
- `RLE_20260831.xlsx` — Assessment criteria per province (36 provinces)
- `Mangrove_RLE.zip` — Province boundaries (shapefile)
- `Individual.assessments.and.associated.DOIs.docx` — Report links with DOIs

### Field mapping notes
The popup component (`map-popup.tsx`) maps field suffixes `_1`, `_2`, `_3` to tags in this order:
- Reduction / Env degradation / Biotic processes: `_1` = Historical (1750), `_2` = Past 50 years, `_3` = Future
- Restricted geographic distribution: `_1` = Extent of occurrence, `_2` = Area of occupancy, `_3` = Threat locations
- Quantitative risk analysis: `_1` = single value

The Excel columns are in a different order (Past, Future, Historical), so we reorder during mapping.

In [ ]:
import pandas as pd
import geopandas as gpd
from docx import Document
from pathlib import Path

RAW_DIR = Path('../../data/raw')
OUT_DIR = Path('../../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load assessment data from Excel

In [ ]:
# Excel has a two-row header (merged group headers + sub-headers).
# Skip both and assign descriptive column names.
df = pd.read_excel(RAW_DIR / 'RLE_20260831.xlsx', header=None, skiprows=2)

df.columns = [
    'province',
    # Reduction in Geographic Distribution
    'red_past50', 'red_future', 'red_hist',
    # Restricted Geographic Distribution
    'res_extent', 'res_area', 'res_threat',
    # Environmental Degradation
    'env_past50', 'env_future', 'env_hist',
    # Distribution of Biotic Processes
    'bio_past50', 'bio_future', 'bio_hist',
    # Quantitative Risk Analysis & Overall
    'qra', 'overall',
]

print(f'Loaded {len(df)} provinces from Excel')
df.head()

## 2. Load province boundaries from shapefile

In [ ]:
gdf = gpd.read_file(f'zip://{RAW_DIR / "Mangrove_RLE.zip"}')
print(f'Loaded {len(gdf)} features from shapefile')
print(f'CRS: {gdf.crs}')
gdf[['unit_name', 'shortname', 'Region', 'Overall']].head(10)

## 3. Parse report links from Word document

In [ ]:
doc = Document(RAW_DIR / 'Individual.assessments.and.associated.DOIs.docx')

reports = []
for para in doc.paragraphs:
    text = para.text.strip()
    if 'doi.org' in text:
        # Each line with a DOI: "Mangroves of the X https://doi.org/..."
        parts = text.rsplit('https://', 1)
        name = parts[0].strip()
        url = 'https://' + parts[1].strip()
        reports.append({'name': f'RLE {name} (pdf)', 'url': url})

reports_df = pd.DataFrame(reports)
print(f'Found {len(reports_df)} reports')
reports_df

## 4. Map Excel columns to layer field names

The frontend popup iterates tags by index and builds field names as `{criterion}_{index+1}`.

For criteria with temporal sub-assessments, the popup tag order is:
- tag[0] → `_1` = **Historical (1750)**
- tag[1] → `_2` = **Past 50 years (1970)**
- tag[2] → `_3` = **Future**

But the Excel column order within each group is: Past 50 yrs, Future, Historical.  
So we remap: Excel col 3 → `_1`, col 1 → `_2`, col 2 → `_3` (within each group).

In [ ]:
rename_map = {
    # Reduction in Geographic Distribution
    'red_hist':    'reduction_in_geographic_distribution_1',   # Historical (1750)
    'red_past50':  'reduction_in_geographic_distribution_2',   # Past 50 years
    'red_future':  'reduction_in_geographic_distribution_3',   # Future
    # Restricted Geographic Distribution
    'res_extent':  'restricted_geographic_distribution_1',     # Extent of occurrence
    'res_area':    'restricted_geographic_distribution_2',     # Area of occupancy
    'res_threat':  'restricted_geographic_distribution_3',     # Threat locations <5
    # Environmental Degradation
    'env_hist':    'environmental_degradation_1',              # Historical (1750)
    'env_past50':  'environmental_degradation_2',              # Past 50 years
    'env_future':  'environmental_degradation_3',              # Future
    # Distribution of Biotic Processes
    'bio_hist':    'distribution_of_biotic_processes_1',       # Historical (1750)
    'bio_past50':  'distribution_of_biotic_processes_2',       # Past 50 years
    'bio_future':  'distribution_of_biotic_processes_3',       # Future
    # Quantitative Risk Analysis
    'qra':         'quantitative_risk_analysis_1',
    # Overall
    'overall':     'overall_assessment',
}

assessments = df.rename(columns=rename_map)
assessments.head()

## 5. Verify assessment categories

The data uses standard IUCN abbreviations: **CR** (Critically Endangered), EN, VU, NT, LC, DD, NE.  
The frontend currently uses `CE` instead of `CR` — that needs to be updated on the frontend side  
along with adding support for `NE` and `NT`.

In [ ]:
criteria_cols = [c for c in assessments.columns if c != 'province']

# Show all unique assessment values
all_values = set()
for col in criteria_cols:
    all_values.update(assessments[col].unique())
print(f'Assessment categories in data: {sorted(all_values)}')
print(f'Standard IUCN categories: CR, DD, EN, LC, NE, NT, VU')

## 6. Join assessments to shapefile

In [ ]:
# Verify province names match between Excel and shapefile
excel_provinces = set(assessments['province'])
shp_provinces = set(gdf['shortname'])

missing_in_shp = excel_provinces - shp_provinces
missing_in_excel = shp_provinces - excel_provinces

if missing_in_shp:
    print(f'WARNING: In Excel but not in shapefile: {missing_in_shp}')
if missing_in_excel:
    print(f'WARNING: In shapefile but not in Excel: {missing_in_excel}')
if not missing_in_shp and not missing_in_excel:
    print(f'All {len(excel_provinces)} province names match between Excel and shapefile')

In [ ]:
layer = gdf.merge(assessments, left_on='shortname', right_on='province', how='left')

# Keep only the fields needed by the frontend
layer_fields = [
    'Region', 'unit_name', 'overall_assessment',
    'reduction_in_geographic_distribution_1',
    'reduction_in_geographic_distribution_2',
    'reduction_in_geographic_distribution_3',
    'restricted_geographic_distribution_1',
    'restricted_geographic_distribution_2',
    'restricted_geographic_distribution_3',
    'environmental_degradation_1',
    'environmental_degradation_2',
    'environmental_degradation_3',
    'distribution_of_biotic_processes_1',
    'distribution_of_biotic_processes_2',
    'distribution_of_biotic_processes_3',
    'quantitative_risk_analysis_1',
    'geometry',
]

layer_out = layer[layer_fields].copy()
print(f'Layer features: {len(layer_out)}')
layer_out.drop(columns='geometry').head()

## 7. Generate backoffice outputs

### ecoregions.csv
Aggregate counts by overall assessment category, matching the existing format:  
`indicator,category,value`

In [ ]:
overall_counts = assessments['overall_assessment'].str.lower().value_counts()

# Use standard IUCN categories
categories = ['dd', 'lc', 'nt', 'vu', 'en', 'cr', 'ne']

ecoregions = pd.DataFrame({
    'indicator': 'ecoregion_asssessment',  # NOTE: triple-s typo matches existing data
    'category': categories,
    'value': [float(overall_counts.get(c, 0)) for c in categories],
})

ecoregions.to_csv(OUT_DIR / 'ecoregions.csv', index=False)
print(f'Saved to {OUT_DIR / "ecoregions.csv"}')
print(f'Total provinces: {ecoregions["value"].sum():.0f}')
ecoregions

### ecoregion_reports.csv

In [ ]:
reports_df.to_csv(OUT_DIR / 'ecoregion_reports.csv', index=False)
print(f'Saved {len(reports_df)} reports to {OUT_DIR / "ecoregion_reports.csv"}')
reports_df

## 8. Export layer for Mapbox upload

Export as GeoJSON. After uploading to Mapbox as a tileset, update the source-layer name
to `ecoregions_data` (or update `hooks.tsx` if using a different name).

In [ ]:
output_path = OUT_DIR / 'iucn_ecoregion_layer.geojson'
layer_out.to_file(output_path, driver='GeoJSON')

print(f'Saved layer to {output_path}')
print(f'Features: {len(layer_out)}')
print(f'Fields: {[c for c in layer_out.columns if c != "geometry"]}')

## 9. Validation

In [ ]:
print('=== Field check ===')
required_fields = [f for f in layer_fields if f != 'geometry']
for f in required_fields:
    assert f in layer_out.columns, f'Missing field: {f}'
print(f'All {len(required_fields)} required fields present')

print(f'\nNull overall_assessment: {layer_out["overall_assessment"].isna().sum()}')

print(f'\n=== Overall assessment distribution ===')
print(layer_out['overall_assessment'].value_counts().sort_index())

print(f'\n=== Spot-check original SE Asian provinces ===')
for name in ['Bay of Bengal', 'Sunda Shelf', 'Andaman', 'South China Sea', 'Western Coral Triangle']:
    row = layer_out[layer_out['unit_name'].str.contains(name, na=False)]
    if not row.empty:
        print(f'  {name}: overall = {row["overall_assessment"].values[0]}')
    else:
        print(f'  {name}: NOT FOUND in layer')

In [ ]:
# Check report matching: the popup strips "Mangroves of" from unit_name
# and uses includes() to find the matching report.
print('=== Report matching check ===')
unmatched = []
for _, row in layer_out.iterrows():
    uname = row['unit_name']
    if pd.isna(uname):
        continue
    trimmed = uname.replace('Mangroves of', '').strip()
    match = reports_df[reports_df['name'].str.contains(trimmed, na=False)]
    if match.empty:
        unmatched.append(uname)

if unmatched:
    print(f'No matching report for {len(unmatched)} provinces (expected for NE provinces):')
    for u in unmatched:
        oa = layer_out[layer_out['unit_name'] == u]['overall_assessment'].values[0]
        print(f'  {u} (overall: {oa})')
else:
    print('All provinces have matching reports')

## Frontend changes required

The data now uses standard IUCN abbreviations. The frontend needs these updates:

1. **CR instead of CE**: Replace all `ce`/`CE` references with `cr`/`CR` in:
   - `types.d.ts` — `CategoryIds` type
   - `constants.ts` — `COLORS` and `LABELS` objects
   - `hooks.tsx` — `OVERALL_ASSESSMENT` paint map

2. **Add NE (Not Evaluated)** and **NT (Near Threatened)** support:
   - Add to `CategoryIds` type
   - Add to `COLORS`: `nt: '#F3AD6A'`, `ne: '#ECECEF'` (NE color already exists)
   - Add to `LABELS`: `nt: 'Near Threatened'`, `ne: 'Not Evaluated'`
   - Add to `OVERALL_ASSESSMENT` paint map in `hooks.tsx`:
     ```js
     const OVERALL_ASSESSMENT = {
       CR: '#EE4D5A',  // was CE
       EN: '#F97B57',
       VU: '#ECDA9A',
       NT: '#F3AD6A',  // new
       LC: '#B4DCAA',
       DD: '#ECECEF',
       NE: '#ECECEF',  // new
     };
     ```
   - Remove the NT filter in `hooks.tsx`: `data?.filter((d) => d.category !== 'nt')`
   - Update popup legend items in `map-popup.tsx` to include all categories